# SSA: Parameter Exploration

### Libruary imports

In [31]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np

from src import ssa_visualization as viz
from src.ssa_simulation import compute_sample_moments, simulate_telegraph
from src.ssa_visualization import show_sample_moments, show_single_trajectory

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
def run_experiment(
    title: str, n_grid=1000, time_step=None, t_end=None,
    show_single_trajectory=False, **kwargs
):
    """Run the Gillespie SSA simulation and plot the resulting sample moments.

    title : str
        The descriptive name of the parameter exploration setup.
    n_grid, time_step, t_end :
        Real-time resampling controls forwarded to compute_sample_moments.
        Use time_step (seconds) for a fixed step, or n_grid for a point count.
    show_single_trajectory : bool
        If True, also show a separate standalone figure of one raw realization
        (gene state, RNA count, and its own deviation product).
    **kwargs : dict
        Model parameters to dynamically override.
    """
    base_params = {
        "k_on": 0.1,
        "k_off": 0.1,
        "k_syn": 10.0,
        "k_deg": 0.2,
        "t0": 0.0,
        "g0": 0,
        "r0": 0,
        "n_sim": 4000,
        "n_rep": 1000
    }
    
    current_params = {**base_params, **kwargs}
    
    data = simulate_telegraph(**current_params)
    moments = compute_sample_moments(
        data, t_end=t_end, n_grid=n_grid, time_step=time_step
    )
    
    show_sample_moments(moments, title=title)
    if show_single_trajectory:
        viz.show_single_trajectory(
            data, trajectory_index=0, title=f"{title} (Single Realization)"
        )

---
### 1. Fast vs. Slow Gene Switching

The **switching regime** is set by how $k_{\text{on}}$ and $k_{\text{off}}$
compare to the mRNA degradation rate $k_{\text{deg}}$.

**Goal:** see how the promoter switching speed shapes mRNA noise.
**Watch:** width of the ⟨R⟩ band ($\sigma_R$) and the strength of the
Gene–RNA covariance.

**1a. Slow switching** ($k_{\text{on}}=0.01,\; k_{\text{off}}=0.02 \ll k_{\text{deg}}=0.2$)

Gene rarely flips, so each cell stays locked ON or OFF for a long time →
strong cell-to-cell variability, **wide $\sigma_R$** and **high G–R covariance** .

In [33]:
run_experiment(
    title="Slow Gene Switching Regime",
    k_on=2.7,
    k_off=1,
    k_syn=89.0,
    show_single_trajectory=True
)

**1b. Fast switching** ($k_{\text{on}}=2.0,\; k_{\text{off}}=4.0 \gg k_{\text{deg}}=0.2$)

Gene flips many times per mRNA lifetime → promoter is effectively averaged
(quasi-constitutive) → **narrow $\sigma_R$**, **near-zero covariance**.
(Note: plot title still reads "Slow" — copy/paste leftover.)

In [34]:
run_experiment(
    title="Slow Gene Switching Regime",
    k_on=2,
    k_off=4
)

---
## 2. High vs. Low Expression Level

Mean mRNA level is governed by the ratio $k_{\text{syn}}/k_{\text{deg}}$.

**Goal:** show how this ratio sets the steady-state ⟨R⟩.

**2a. High expression** ($k_{\text{syn}}/k_{\text{deg}} = 40/0.5 = 80$)

Expect a **high steady-state ⟨R⟩** with correspondingly large absolute fluctuations.

In [35]:
run_experiment(
    title="High Synthesis and Low Degradation Rates",
    k_syn=40.0,
    k_deg=0.5
)


**2b. Low expression** ($k_{\text{syn}}/k_{\text{deg}} = 2/1 = 2$)

Expect a **low ⟨R⟩** (few molecules), where discreteness makes relative noise large.

In [36]:
run_experiment(
    title="Low Synthesis and Degradation Rates",
    k_syn=2.0,
    k_deg=1.0
)


---
## 3. Effect of Initial Conditions

Same rates, different starting state $(g_0, r_0)$.

**Goal:** confirm all initial conditions **relax to the same steady state**;
the transient just shows how the system forgets its start.

**3a. Cold start** ($g_0=0,\; r_0=0$)

Gene and mRNA both empty → ⟨R⟩ **builds up** from zero toward steady state.

In [37]:
run_experiment(
    title="Starting with Zero Active Genes and Zero mRNA Molecules",
    k_on=0.05,
    k_off=0.05,
    k_syn=10.0,
    k_deg=0.2,
    n_sim=4000,
    n_rep=300,
    t0=0.0,
    g0=0,  
    r0=0
)

**3b. Warm start** ($g_0=1,\; r_0=50$)

Gene ON and mRNA already abundant → ⟨R⟩ **relaxes down** to the same steady state.

In [38]:
run_experiment(
    title="Starting with One Active Gene and 50 mRNA Molecules",
    k_on=0.05,
    k_off=0.05,
    k_syn=10.0,
    k_deg=0.2,
    n_sim=4000,
    n_rep=300,
    t0=0.0,
    g0=1,  
    r0=50
)

**3c. Mismatched start** ($g_0=0,\; r_0=50$)

Gene OFF but mRNA abundant → the surplus mRNA **decays** until equilibrium;
all three runs (3a–3c) converge to the same level.

In [39]:
run_experiment(
    title="Starting with Zero Active Genes and 50 mRNA Molecules",
    k_on=0.05,
    k_off=0.05,
    k_syn=10.0,
    k_deg=0.2,
    n_sim=4000,
    n_rep=300,
    t0=0.0,
    g0=0,  
    r0=50
)

---
## 4. Effect of Numerical Parameters

These knobs change the **estimate quality**.

**Goal:** show how ensemble size and run length affect the sampled moments.

**4a. Ensemble size $n_{\text{rep}}$** (50 vs 1000 trajectories)

More trajectories → less sampling error → **smoother moment curves / tighter bands**.
Small $n_{\text{rep}}$ looks jagged.

In [40]:
run_experiment(
    title="Numerical Effects: Low Ensemble Size (n_rep=50)",
    k_on=0.05,
    n_sim=3000,
    n_rep=50,
)

In [41]:
run_experiment(
    title="Numerical Effects: High Ensemble Size (n_rep=1000)",
    k_on=0.05,
    n_sim=3000,
    n_rep=1000,
)

**4b. Run length $n_{\text{sim}}$** (400 vs 4000 events)

$n_{\text{sim}}$ sets how far in time each trajectory reaches. Short runs may
**not reach steady state**; long runs show the full relaxation and plateau.

In [42]:
run_experiment(
    title="Numerical Effects: Short Simulation (n_sim=400)",
    k_on=0.05,
    n_sim=400,
    n_rep=300,
)

In [43]:
run_experiment(
    title="Numerical Effects: Long Simulation (n_sim=4000)",
    k_on=0.05,
    n_sim=4000,
    n_rep=300,
)